# Step 8 Text-Only Refresh

This notebook explores whether a metadata-free branch can add a useful second view of the data.
The DeBERTa runs are left out here because the local snapshot was not complete on this machine.


## Environment Check

The setup cell keeps the path logic consistent with the earlier notebooks.


In [1]:
from pathlib import Path
import json
import subprocess
import sys

import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)

EXPECTED_ENV = 'usc-nlp'
if EXPECTED_ENV not in sys.executable.lower():
    raise RuntimeError(f'This notebook should run inside the {EXPECTED_ENV} conda environment. Current executable: {sys.executable}')


def detect_project_dir(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'group17_runtime.py').exists():
            return candidate
    raise FileNotFoundError('Could not find the project root from the notebook location.')


def find_latest_run(root: Path, prefix: str) -> Path:
    candidates = [path for path in root.iterdir() if path.is_dir() and path.name.startswith(prefix)]
    if not candidates:
        raise FileNotFoundError(f'No run found in {root} with prefix {prefix!r}.')
    return max(candidates, key=lambda path: path.stat().st_mtime)


def read_json(path: Path) -> dict:
    with path.open('r', encoding='utf-8') as handle:
        return json.load(handle)


def run_command(cmd: list[str]) -> None:
    print('Running command:')
    print(' '.join(cmd))
    subprocess.run(cmd, check=True)


PROJECT_DIR = detect_project_dir(Path.cwd().resolve())
print(f'Project directory: {PROJECT_DIR}')
print(f'Python executable: {sys.executable}')


Project directory: D:\CS544-Group17-Project
Python executable: f:\anaconda\envs\usc-nlp\python.exe


## Step 8 Plan

These are the text-only runs that were worth keeping for the final line.


In [2]:
STEP8_SCRIPT = PROJECT_DIR / 'group17_step8_textonly_refresh.py'
OUTPUT_ROOT = PROJECT_DIR / 'step8_outputs'
RUN_ID = 'exp_s81_textonly_refresh_submission'
EXPERIMENT_IDS = [
    's81_bertweet_textonly_v1',
    's82_twitter_roberta_textonly_v1',
    's84_bertweet_textonly_long_v1',
    's85_twitter_roberta_textonly_long_v1',
]

step8_cmd = [sys.executable, str(STEP8_SCRIPT), '--run-id', RUN_ID]
for experiment_id in EXPERIMENT_IDS:
    step8_cmd.extend(['--experiment-id', experiment_id])

pd.DataFrame({'experiment_id': EXPERIMENT_IDS})


,experiment_id
0,s81_bertweet_textonly_v1
1,s82_twitter_roberta_textonly_v1
2,s84_bertweet_textonly_long_v1
3,s85_twitter_roberta_textonly_long_v1


## Launch Step 8

This creates a fresh Step 8 campaign with only the text-only runs we want to report.


In [3]:
run_command(step8_cmd)
STEP8_RUN_DIR = find_latest_run(OUTPUT_ROOT, RUN_ID)
print(f'Step 8 run directory: {STEP8_RUN_DIR}')


Running command:
f:\anaconda\envs\usc-nlp\python.exe D:\CS544-Group17-Project\group17_step8_textonly_refresh.py --run-id exp_s81_textonly_refresh_submission --experiment-id s81_bertweet_textonly_v1 --experiment-id s82_twitter_roberta_textonly_v1 --experiment-id s84_bertweet_textonly_long_v1 --experiment-id s85_twitter_roberta_textonly_long_v1
Step 8 run directory: D:\CS544-Group17-Project\step8_outputs\exp_s81_textonly_refresh_submission_local_20260424_084702


## Campaign Summary

This file is the quickest way to compare the four text-only experiments.


In [4]:
step8_campaign = pd.read_csv(STEP8_RUN_DIR / 'campaign_summary.csv')
display(step8_campaign)


,experiment_id,stage,model_name,train_name,cv_F1,cv_F1_std,cv_Precision,cv_Recall,cv_Accuracy,tuned_F1,best_threshold,output_dir,status
0,s81_bertweet_textonly_v1,core,vinai/bertweet-base,core text-only,0.809336,0.016985,0.835032,0.786917,0.842329,0.809347,0.50,D:\CS544-Group17-Project\step8_outputs\exp_s81_textonly_refresh_submission_local_20260424_084702\s81_bertweet_texton...,completed
1,s82_twitter_roberta_textonly_v1,core,cardiffnlp/twitter-roberta-base,core text-only,0.807905,0.013028,0.841596,0.777194,0.842863,0.809153,0.48,D:\CS544-Group17-Project\step8_outputs\exp_s81_textonly_refresh_submission_local_20260424_084702\s82_twitter_roberta...,completed
2,s84_bertweet_textonly_long_v1,extended,vinai/bertweet-base,extended text-only,0.808142,0.014625,0.823216,0.794434,0.839797,0.811348,0.72,D:\CS544-Group17-Project\step8_outputs\exp_s81_textonly_refresh_submission_local_20260424_084702\s84_bertweet_texton...,completed
3,s85_twitter_roberta_textonly_long_v1,extended,cardiffnlp/twitter-roberta-base,extended text-only,0.806815,0.013799,0.839434,0.776877,0.841664,0.807761,0.47,D:\CS544-Group17-Project\step8_outputs\exp_s81_textonly_refresh_submission_local_20260424_084702\s85_twitter_roberta...,completed


## Selected Step 8 Metrics

The long BERTweet run is the one that later becomes the Step 9 partner.


In [5]:
rows = []
for experiment_id in EXPERIMENT_IDS:
    summary_path = STEP8_RUN_DIR / experiment_id / f'{experiment_id}_summary.json'
    summary = read_json(summary_path)
    tuned = summary['tuned_threshold']
    rows.append({
        'experiment_id': experiment_id,
        'model_name': summary['model_name'],
        'train_name': summary['train_name'],
        'best_threshold': summary['best_threshold'],
        'F1': tuned['F1'],
        'Precision': tuned['Precision'],
        'Recall': tuned['Recall'],
        'Accuracy': tuned['Accuracy'],
    })

step8_results = pd.DataFrame(rows).sort_values('F1', ascending=False).reset_index(drop=True)
display(step8_results)


,experiment_id,model_name,train_name,best_threshold,F1,Precision,Recall,Accuracy
0,s84_bertweet_textonly_long_v1,vinai/bertweet-base,extended text-only,0.72,0.811348,0.845683,0.779693,0.845795
1,s81_bertweet_textonly_v1,vinai/bertweet-base,core text-only,0.50,0.809347,0.833112,0.786901,0.842330
2,s82_twitter_roberta_textonly_v1,cardiffnlp/twitter-roberta-base,core text-only,0.48,0.809153,0.839111,0.781260,0.843263
3,s85_twitter_roberta_textonly_long_v1,cardiffnlp/twitter-roberta-base,extended text-only,0.47,0.807761,0.834335,0.782827,0.841530
